In [1]:
import pandas as pd
import requests
import os

# --- Configuración ---
# Diccionario con las temporadas y sus URLs
seasons = {
    "2022-2023": "https://fbref.com/en/comps/12/2022-2023/schedule/2022-2023-La-Liga-Scores-and-Fixtures",
    "2023-2024": "https://fbref.com/en/comps/12/2023-2024/schedule/2023-2024-La-Liga-Scores-and-Fixtures",
    "2024-2025": "https://fbref.com/en/comps/12/2024-2025/schedule/2024-2025-La-Liga-Scores-and-Fixtures",
}

# Ruta donde guardaremos los datos crudos
# ../ significa "subir un nivel" desde la carpeta /notebooks a la raíz del proyecto
path_raw = "../data/raw/"

# --- Proceso de Descarga ---
print("🤖 Iniciando la descarga automática de datos...")

# Nos aseguramos de que la carpeta de destino exista
os.makedirs(path_raw, exist_ok=True)

for season, url in seasons.items():
    try:
        print(f"Procesando temporada {season}...")
        
        # Leemos la tabla directamente desde la URL usando Pandas
        df = pd.read_html(url)[0]
        
        # Definimos el nombre del archivo de salida
        file_name = f"laliga_scores_{season}.csv"
        full_path = os.path.join(path_raw, file_name)
        
        # Guardamos la tabla como un archivo CSV
        df.to_csv(full_path, index=False)
        
        print(f"✅ Temporada {season} guardada con éxito en: {full_path}")

    except Exception as e:
        print(f"❌ Error al descargar la temporada {season}: {e}")

print("\n🎉 ¡Descarga completada!")

🤖 Iniciando la descarga automática de datos...
Procesando temporada 2022-2023...
✅ Temporada 2022-2023 guardada con éxito en: ../data/raw/laliga_scores_2022-2023.csv
Procesando temporada 2023-2024...
✅ Temporada 2023-2024 guardada con éxito en: ../data/raw/laliga_scores_2023-2024.csv
Procesando temporada 2024-2025...
✅ Temporada 2024-2025 guardada con éxito en: ../data/raw/laliga_scores_2024-2025.csv

🎉 ¡Descarga completada!


In [3]:
# Cargamos el dataset de la temporada más reciente en un DataFrame de Pandas
df_2425 = pd.read_csv('../data/raw/laliga_scores_2024-2025.csv')

# Mostramos las primeras 5 filas para ver cómo se ven los datos
df_2425.head()

,Wk,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes
0,1.0,Thu,2024-08-15,19:00,Athletic Club,0.3,1–1,0.8,Getafe,47845.0,San Mamés,Alejandro Muñíz,Match Report,NaN
1,1.0,Thu,2024-08-15,21:30,Betis,1.4,1–1,1.6,Girona,54084.0,Estadio Benito Villamarín,Miguel Ángel Ortiz Arias,Match Report,NaN
2,1.0,Fri,2024-08-16,19:00,Celta Vigo,0.8,2–1,1.6,Alavés,22477.0,Estadio Abanca Balaídos,Alejandro Quintero,Match Report,NaN
3,1.0,Fri,2024-08-16,20:30,Las Palmas,1.4,2–2,1.8,Sevilla,24843.0,Estadio de Gran Canaria,Francisco Hernández,Match Report,NaN
4,1.0,Sat,2024-08-17,19:00,Osasuna,1.7,1–1,1.0,Leganés,19561.0,Estadio El Sadar,Juan Pulido,Match Report,NaN


In [4]:
# Muestra un resumen técnico del DataFrame
df_2425.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Wk            380 non-null    float64
 1   Day           380 non-null    object 
 2   Date          380 non-null    object 
 3   Time          380 non-null    object 
 4   Home          380 non-null    object 
 5   xG            380 non-null    float64
 6   Score         380 non-null    object 
 7   xG.1          380 non-null    float64
 8   Away          380 non-null    object 
 9   Attendance    380 non-null    float64
 10  Venue         380 non-null    object 
 11  Referee       380 non-null    object 
 12  Match Report  380 non-null    object 
 13  Notes         0 non-null      float64
dtypes: float64(5), object(9)
memory usage: 46.7+ KB


In [5]:
# 1. Eliminar columnas innecesarias
df_clean = df_2425.drop(columns=["Match Report", "Notes"])

# 2. Eliminar filas vacías
df_clean = df_clean.dropna(subset=["Home", "Away", "Score"])

# 3. Separar marcador
df_clean[["HomeGoals", "AwayGoals"]] = df_clean["Score"].str.split("–", expand=True)
df_clean["HomeGoals"] = df_clean["HomeGoals"].astype(int)
df_clean["AwayGoals"] = df_clean["AwayGoals"].astype(int)

# 4. Convertir fecha a datetime
df_clean["Date"] = pd.to_datetime(df_clean["Date"])

# Revisar cómo queda
df_clean.head()


,Wk,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,HomeGoals,AwayGoals
0,1.0,Thu,2024-08-15,19:00,Athletic Club,0.3,1–1,0.8,Getafe,47845.0,San Mamés,Alejandro Muñíz,1,1
1,1.0,Thu,2024-08-15,21:30,Betis,1.4,1–1,1.6,Girona,54084.0,Estadio Benito Villamarín,Miguel Ángel Ortiz Arias,1,1
2,1.0,Fri,2024-08-16,19:00,Celta Vigo,0.8,2–1,1.6,Alavés,22477.0,Estadio Abanca Balaídos,Alejandro Quintero,2,1
3,1.0,Fri,2024-08-16,20:30,Las Palmas,1.4,2–2,1.8,Sevilla,24843.0,Estadio de Gran Canaria,Francisco Hernández,2,2
4,1.0,Sat,2024-08-17,19:00,Osasuna,1.7,1–1,1.0,Leganés,19561.0,Estadio El Sadar,Juan Pulido,1,1


In [7]:
def clean_season_data(df):
    """
    Esta función toma un DataFrame de temporada crudo y aplica la limpieza.
    """
    # 1. Eliminar columnas innecesarias (usamos una copia para evitar advertencias)
    df_clean = df.drop(columns=["Match Report", "Notes"]).copy()
    
    # 2. Eliminar filas vacías en columnas clave
    df_clean.dropna(subset=["Home", "Away", "Score"], inplace=True)
    
    # 3. Separar marcador en goles (HomeGoals y AwayGoals)
    # Usamos .str.split() y convertimos a número, manejando errores
    score_split = df_clean["Score"].str.split('–', expand=True)
    df_clean["HomeGoals"] = pd.to_numeric(score_split[0], errors='coerce')
    df_clean["AwayGoals"] = pd.to_numeric(score_split[1], errors='coerce')
    
    # 4. Convertir fecha a datetime
    df_clean["Date"] = pd.to_datetime(df_clean["Date"])
    
    # Eliminamos la columna original de "Score" que ya no necesitamos
    df_clean.drop(columns=["Score"], inplace=True)
    
    return df_clean

In [8]:
# Lista de los nombres de archivo que descargamos
files = [
    '../data/raw/laliga_scores_2022-2023.csv',
    '../data/raw/laliga_scores_2023-2024.csv',
    '../data/raw/laliga_scores_2024-2025.csv'
]

# Creamos una lista para guardar los DataFrames ya limpios
list_of_clean_dfs = []

print("Procesando y combinando todas las temporadas...")

for file in files:
    # Cargamos el archivo crudo
    df_raw = pd.read_csv(file)
    # Aplicamos nuestra función de limpieza
    df_clean = clean_season_data(df_raw)
    # Añadimos el DataFrame limpio a nuestra lista
    list_of_clean_dfs.append(df_clean)

# Combinamos todos los DataFrames de la lista en uno solo
df_master = pd.concat(list_of_clean_dfs, ignore_index=True)

print("\n¡Proceso completado!")
# Verificamos el resultado final
print("Dimensiones del DataFrame maestro:", df_master.shape)
display(df_master.head()) # Muestra las primeras filas
display(df_master.tail())  # Muestra las últimas filas

Procesando y combinando todas las temporadas...

¡Proceso completado!
Dimensiones del DataFrame maestro: (1140, 13)


,Wk,Day,Date,Time,Home,xG,xG.1,Away,Attendance,Venue,Referee,HomeGoals,AwayGoals
0,1.0,Fri,2022-08-12,21:00,Osasuna,1.5,0.9,Sevilla,18536.0,Estadio El Sadar,Carlos del Cerro,2,1
1,1.0,Sat,2022-08-13,17:00,Celta Vigo,0.4,1.1,Espanyol,13859.0,Estadio de Balaídos,Miguel Ángel Ortiz Arias,2,2
2,1.0,Sat,2022-08-13,19:00,Valladolid,1.0,1.5,Villarreal,17543.0,Estadio Municipal José Zorrilla,Mario Melero,0,3
3,1.0,Sat,2022-08-13,21:00,Barcelona,1.9,0.5,Rayo Vallecano,81104.0,Camp Nou,Alejandro Hernández,0,0
4,1.0,Sun,2022-08-14,17:30,Cádiz,0.2,1.7,Real Sociedad,16570.0,Estadio Nuevo Mirandilla,Isidro Díaz de Mera,0,1


,Wk,Day,Date,Time,Home,xG,xG.1,Away,Attendance,Venue,Referee,HomeGoals,AwayGoals
1135,38.0,Sat,2025-05-24,21:00,Alavés,2.0,1.6,Osasuna,19274.0,Estadio de Mendizorroza,Guillermo Cuadra,1,1
1136,38.0,Sat,2025-05-24,21:00,Getafe,0.4,1.5,Celta Vigo,12862.0,Coliseum Alfonso Pérez,Juan Martínez,1,2
1137,38.0,Sun,2025-05-25,14:00,Girona,0.0,2.7,Atlético Madrid,11546.0,Estadi Municipal de Montilivi,Jesús Gil,0,4
1138,38.0,Sun,2025-05-25,16:15,Villarreal,1.1,1.5,Sevilla,17758.0,Estadio de la Cerámica,Javier Alberola,4,2
1139,38.0,Sun,2025-05-25,21:00,Athletic Club,1.2,3.5,Barcelona,50231.0,San Mamés,Pablo González,0,3


In [9]:
import numpy as np

# Creamos una función para determinar el resultado
def get_match_result(row):
    if row['HomeGoals'] > row['AwayGoals']:
        return 'H'  # Victoria Local (Home win)
    elif row['HomeGoals'] < row['AwayGoals']:
        return 'A'  # Victoria Visitante (Away win)
    else:
        return 'D'  # Empate (Draw)

# Aplicamos la función a cada fila para crear la nueva columna 'Result'
df_master['Result'] = df_master.apply(get_match_result, axis=1)

# Revisamos cómo quedó, mostrando la nueva columna al final
display(df_master.head())

,Wk,Day,Date,Time,Home,xG,xG.1,Away,Attendance,Venue,Referee,HomeGoals,AwayGoals,Result
0,1.0,Fri,2022-08-12,21:00,Osasuna,1.5,0.9,Sevilla,18536.0,Estadio El Sadar,Carlos del Cerro,2,1,H
1,1.0,Sat,2022-08-13,17:00,Celta Vigo,0.4,1.1,Espanyol,13859.0,Estadio de Balaídos,Miguel Ángel Ortiz Arias,2,2,D
2,1.0,Sat,2022-08-13,19:00,Valladolid,1.0,1.5,Villarreal,17543.0,Estadio Municipal José Zorrilla,Mario Melero,0,3,A
3,1.0,Sat,2022-08-13,21:00,Barcelona,1.9,0.5,Rayo Vallecano,81104.0,Camp Nou,Alejandro Hernández,0,0,D
4,1.0,Sun,2022-08-14,17:30,Cádiz,0.2,1.7,Real Sociedad,16570.0,Estadio Nuevo Mirandilla,Isidro Díaz de Mera,0,1,A


In [10]:
# Agrupamos por equipo local y sumamos sus goles
home_goals = df_master.groupby('Home')['HomeGoals'].sum().sort_values(ascending=False)

print("--- Equipos más goleadores en CASA ---")
print(home_goals.head(5)) # Mostramos el top 5

# Hacemos lo mismo para los equipos visitantes
away_goals = df_master.groupby('Away')['AwayGoals'].sum().sort_values(ascending=False)

print("\n--- Equipos más goleadores FUERA de casa ---")
print(away_goals.head(5))

--- Equipos más goleadores en CASA ---
Home
Real Madrid        137
Barcelona          132
Atlético Madrid    125
Girona             115
Villarreal         115
Name: HomeGoals, dtype: int64

--- Equipos más goleadores FUERA de casa ---
Away
Barcelona          119
Real Madrid        103
Atlético Madrid     83
Villarreal          80
Girona              72
Name: AwayGoals, dtype: int64
